In [11]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU count: 2
0 Tesla T4
VRAM: 14.56 GB
1 Tesla T4
VRAM: 14.56 GB


In [12]:
# ============================================================
# MYRAA KAGGLE GPU WORKER
# CELL 1 — CONFIGURATION
# ============================================================

import os
import json
import time
from pathlib import Path

# ------------------------------------------------------------
# Identity
# ------------------------------------------------------------

WORKER_NAME = "myraa-kaggle-t4x2"

WORKER_VERSION = "1.0.0"

PROVIDER = "kaggle"

# ------------------------------------------------------------
# Directories
# ------------------------------------------------------------

BASE_DIR = Path("/kaggle/working/myraa_worker")

JOBS_DIR = BASE_DIR / "jobs"
INPUT_DIR = BASE_DIR / "inputs"
OUTPUT_DIR = BASE_DIR / "outputs"
LOG_DIR = BASE_DIR / "logs"
CACHE_DIR = BASE_DIR / "cache"
MODEL_DIR = BASE_DIR / "models"

for directory in [
    BASE_DIR,
    JOBS_DIR,
    INPUT_DIR,
    OUTPUT_DIR,
    LOG_DIR,
    CACHE_DIR,
    MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

# Large enough to make multi-GPU sharding meaningful.
#
# If downloading this model is too slow for your first run,
# temporarily change it to:
#
# Qwen/Qwen2.5-7B-Instruct
#
# But 7B will generally fit on one T4, so it is NOT a good
# test of multi-GPU model sharding.

MODEL_ID = os.getenv(
    "MYRAA_MODEL_ID",
    "Qwen/Qwen2.5-32B-Instruct"
)

# 4-bit is required for the 32B-class test on this hardware.
USE_4BIT = True

# T4 is best treated as FP16 hardware.
TORCH_DTYPE = "float16"

# ------------------------------------------------------------
# Generation
# ------------------------------------------------------------

MAX_NEW_TOKENS = 512

TEMPERATURE = 0.7

TOP_P = 0.9

DO_SAMPLE = True

# ------------------------------------------------------------
# Safety limits
# ------------------------------------------------------------

MAX_INPUT_CHARS = 100_000

MAX_OUTPUT_TOKENS = 2_048

# Never allow arbitrary code execution through job input.
ALLOW_ARBITRARY_CODE = False

# ------------------------------------------------------------
# Print configuration
# ------------------------------------------------------------

print("=" * 70)
print("MYRAA KAGGLE GPU WORKER")
print("=" * 70)

print("Worker:", WORKER_NAME)
print("Version:", WORKER_VERSION)
print("Provider:", PROVIDER)

print("\nModel:", MODEL_ID)
print("4-bit:", USE_4BIT)
print("dtype:", TORCH_DTYPE)

print("\nDirectories:")
print("BASE:", BASE_DIR)
print("JOBS:", JOBS_DIR)
print("INPUT:", INPUT_DIR)
print("OUTPUT:", OUTPUT_DIR)
print("CACHE:", CACHE_DIR)

MYRAA KAGGLE GPU WORKER
Worker: myraa-kaggle-t4x2
Version: 1.0.0
Provider: kaggle

Model: Qwen/Qwen2.5-32B-Instruct
4-bit: True
dtype: float16

Directories:
BASE: /kaggle/working/myraa_worker
JOBS: /kaggle/working/myraa_worker/jobs
INPUT: /kaggle/working/myraa_worker/inputs
OUTPUT: /kaggle/working/myraa_worker/outputs
CACHE: /kaggle/working/myraa_worker/cache


In [13]:
# ============================================================
# CELL 7 — MYRAA MODEL LOADER
# Qwen 32B / 4-bit / Multi-GPU
# ============================================================

import os
import gc
import json
import time
import importlib.util

# ------------------------------------------------------------
# CUDA allocator configuration
# ------------------------------------------------------------

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


# ============================================================
# 1. BASIC CONFIG
# ============================================================

MODEL_ID = globals().get(
    "MODEL_ID",
    os.getenv(
        "MYRAA_MODEL_ID",
        "Qwen/Qwen2.5-32B-Instruct"
    )
)

USE_4BIT = bool(
    globals().get(
        "USE_4BIT",
        True
    )
)

TORCH_DTYPE_NAME = str(
    globals().get(
        "TORCH_DTYPE",
        "float16"
    )
).lower()


# ------------------------------------------------------------
# Resolve dtype
# ------------------------------------------------------------

if TORCH_DTYPE_NAME in (
    "float16",
    "fp16",
    "torch.float16"
):

    TORCH_DTYPE = torch.float16

elif TORCH_DTYPE_NAME in (
    "bfloat16",
    "bf16",
    "torch.bfloat16"
):

    TORCH_DTYPE = torch.bfloat16

else:

    print(
        f"Unknown dtype: {TORCH_DTYPE_NAME}"
    )

    print(
        "Falling back to float16."
    )

    TORCH_DTYPE_NAME = "float16"

    TORCH_DTYPE = torch.float16


# ============================================================
# 2. HARDWARE
# ============================================================

print("=" * 70)
print("MYRAA — MODEL LOADER")
print("=" * 70)

CUDA_AVAILABLE = torch.cuda.is_available()

GPU_COUNT = (
    torch.cuda.device_count()
    if CUDA_AVAILABLE
    else 0
)

print(
    f"CUDA available : {CUDA_AVAILABLE}"
)

print(
    f"GPU count      : {GPU_COUNT}"
)

if not CUDA_AVAILABLE:

    raise RuntimeError(
        "CUDA is not available."
    )

if GPU_COUNT < 2:

    raise RuntimeError(
        "MYRAA requires at least 2 GPUs."
    )


for i in range(GPU_COUNT):

    props = (
        torch.cuda.get_device_properties(i)
    )

    print(
        f"GPU {i}: "
        f"{props.name} | "
        f"{props.total_memory / (1024**3):.2f} GB | "
        f"compute {props.major}.{props.minor}"
    )


print("=" * 70)

print(
    f"Model         : {MODEL_ID}"
)

print(
    f"4-bit config  : {USE_4BIT}"
)

print(
    f"dtype         : {TORCH_DTYPE_NAME}"
)

print("=" * 70)


# ============================================================
# 3. VERIFY BITSANDBYTES
# ============================================================

print(
    "Checking bitsandbytes..."
)

BNB_SPEC = importlib.util.find_spec(
    "bitsandbytes"
)

BITSANDBYTES_AVAILABLE = (
    BNB_SPEC is not None
)

print(
    "bitsandbytes module:",
    BITSANDBYTES_AVAILABLE
)


# ------------------------------------------------------------
# Actually import it.
#
# find_spec() alone is not enough.
# ------------------------------------------------------------

BITSANDBYTES_IMPORT_OK = False
BITSANDBYTES_VERSION = None
BITSANDBYTES_ERROR = None

if BITSANDBYTES_AVAILABLE:

    try:

        import bitsandbytes as bnb

        BITSANDBYTES_IMPORT_OK = True

        BITSANDBYTES_VERSION = getattr(
            bnb,
            "__version__",
            "unknown"
        )

        print(
            "bitsandbytes version:",
            BITSANDBYTES_VERSION
        )

    except Exception as exc:

        BITSANDBYTES_IMPORT_OK = False

        BITSANDBYTES_ERROR = str(
            exc
        )

        print(
            "bitsandbytes import FAILED:"
        )

        print(
            BITSANDBYTES_ERROR
        )


print("=" * 70)


# ============================================================
# 4. 4-BIT REQUIREMENT
# ============================================================

if USE_4BIT and not BITSANDBYTES_IMPORT_OK:

    print("=" * 70)
    print("4-BIT MODEL LOAD BLOCKED")
    print("=" * 70)

    print(
        "The current Python kernel cannot successfully "
        "import bitsandbytes."
    )

    print()
    print(
        "This is NOT a GPU/sharding failure."
    )

    print(
        "Transformers requires bitsandbytes for "
        "4-bit NF4 quantization."
    )

    print()
    print(
        "Do NOT continue to model loading."
    )

    print("=" * 70)

    MODEL_SHARDING_VERIFIED = False

    LLM_INFERENCE_VERIFIED = False

    MYRAA_MULTI_GPU_RESULT = {

        "status":
            "blocked",

        "model":
            MODEL_ID,

        "configured_4bit":
            USE_4BIT,

        "actual_4bit":
            False,

        "bitsandbytes_available":
            BITSANDBYTES_AVAILABLE,

        "bitsandbytes_import_ok":
            BITSANDBYTES_IMPORT_OK,

        "bitsandbytes_version":
            BITSANDBYTES_VERSION,

        "bitsandbytes_error":
            BITSANDBYTES_ERROR,

        "model_sharding_verified":
            False,

        "llm_inference_verified":
            False,

        "reason":
            "bitsandbytes cannot be imported "
            "by the current Python kernel."
    }

    globals()[
        "MODEL_SHARDING_VERIFIED"
    ] = False

    globals()[
        "LLM_INFERENCE_VERIFIED"
    ] = False

    globals()[
        "MYRAA_MULTI_GPU_RESULT"
    ] = (
        MYRAA_MULTI_GPU_RESULT
    )

    print(
        json.dumps(
            MYRAA_MULTI_GPU_RESULT,
            indent=2
        )
    )

else:

    # ========================================================
    # 5. CLEAN MEMORY
    # ========================================================

    print(
        "Cleaning GPU memory..."
    )

    gc.collect()

    for i in range(GPU_COUNT):

        try:

            with torch.cuda.device(i):

                torch.cuda.empty_cache()

                torch.cuda.ipc_collect()

        except Exception:
            pass


    print(
        "GPU memory cleanup complete."
    )

    print("=" * 70)


    # ========================================================
    # 6. TOKENIZER
    # ========================================================

    print(
        f"Initializing tokenizer for model: "
        f"{MODEL_ID}..."
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_ID,
            trust_remote_code=True
        )
    )

    if tokenizer.pad_token is None:

        tokenizer.pad_token = (
            tokenizer.eos_token
        )

    print(
        "Tokenizer ready."
    )

    print("=" * 70)


    # ========================================================
    # 7. QUANTIZATION
    # ========================================================

    quantization_config = None

    if USE_4BIT:

        print(
            "Configuring 4-bit NF4 quantization..."
        )

        quantization_config = (
            BitsAndBytesConfig(

                load_in_4bit=True,

                bnb_4bit_quant_type="nf4",

                bnb_4bit_use_double_quant=True,

                bnb_4bit_compute_dtype=(
                    TORCH_DTYPE
                )
            )
        )

    else:

        print(
            "4-bit quantization disabled."
        )


    print("=" * 70)


    # ========================================================
    # 8. GPU MEMORY MAP
    # ========================================================

    MAX_MEMORY = {}

    for i in range(GPU_COUNT):

        total_gb = (
            torch.cuda
            .get_device_properties(i)
            .total_memory
            / (1024 ** 3)
        )

        # Leave 1 GB headroom.
        usable_gb = max(
            1,
            int(
                total_gb - 1.0
            )
        )

        MAX_MEMORY[i] = (
            f"{usable_gb}GiB"
        )


    print(
        "GPU memory map:"
    )

    print("-" * 70)

    for key, value in (
        MAX_MEMORY.items()
    ):

        print(
            f"GPU {key}: {value}"
        )

    print("-" * 70)

    print(
        "CPU offload: DISABLED"
    )

    print("=" * 70)


    # ========================================================
    # 9. MODEL LOAD
    # ========================================================

    print(
        f"Loading model: {MODEL_ID}..."
    )

    print(
        "device_map = auto"
    )

    print(
        "4-bit multi-GPU loading enabled."
    )

    print()

    LOAD_START = time.perf_counter()

    model = None

    try:

        load_kwargs = {

            "device_map":
                "auto",

            "max_memory":
                MAX_MEMORY,

            "dtype":
                TORCH_DTYPE,

            "low_cpu_mem_usage":
                True,

            "trust_remote_code":
                True,
        }


        if quantization_config is not None:

            load_kwargs[
                "quantization_config"
            ] = (
                quantization_config
            )


        model = (
            AutoModelForCausalLM
            .from_pretrained(
                MODEL_ID,
                **load_kwargs
            )
        )


    except Exception as exc:

        LOAD_TIME = (
            time.perf_counter()
            - LOAD_START
        )

        print()
        print("=" * 70)
        print("MODEL LOAD FAILED")
        print("=" * 70)

        print(
            f"Error type: "
            f"{type(exc).__name__}"
        )

        print(
            f"Error: {exc}"
        )

        print(
            f"Load time: "
            f"{LOAD_TIME:.2f} sec"
        )

        print("=" * 70)


        MODEL_SHARDING_VERIFIED = False

        LLM_INFERENCE_VERIFIED = False

        MYRAA_MULTI_GPU_RESULT = {

            "status":
                "load_failed",

            "model":
                MODEL_ID,

            "configured_4bit":
                USE_4BIT,

            "actual_4bit":
                (
                    USE_4BIT
                    and
                    BITSANDBYTES_IMPORT_OK
                ),

            "bitsandbytes_available":
                BITSANDBYTES_AVAILABLE,

            "bitsandbytes_import_ok":
                BITSANDBYTES_IMPORT_OK,

            "bitsandbytes_version":
                BITSANDBYTES_VERSION,

            "dtype":
                TORCH_DTYPE_NAME,

            "gpu_count":
                GPU_COUNT,

            "gpu_memory_map":
                MAX_MEMORY,

            "cpu_offload_enabled":
                False,

            "model_sharding_verified":
                False,

            "llm_inference_verified":
                False,

            "error_type":
                type(exc).__name__,

            "error":
                str(exc),

            "load_time_seconds":
                round(
                    LOAD_TIME,
                    3
                )
        }


        globals()[
            "MODEL_SHARDING_VERIFIED"
        ] = False

        globals()[
            "LLM_INFERENCE_VERIFIED"
        ] = False

        globals()[
            "MYRAA_MULTI_GPU_RESULT"
        ] = (
            MYRAA_MULTI_GPU_RESULT
        )


        print(
            json.dumps(
                MYRAA_MULTI_GPU_RESULT,
                indent=2
            )
        )


    else:

        # ====================================================
        # 10. MODEL LOADED
        # ====================================================

        LOAD_TIME = (
            time.perf_counter()
            - LOAD_START
        )

        print()
        print("=" * 70)
        print(
            "MODEL LOADED SUCCESSFULLY"
        )
        print("=" * 70)

        print(
            f"Load time: "
            f"{LOAD_TIME:.2f} sec"
        )

        print("=" * 70)


        # ====================================================
        # 11. DEVICE MAP
        # ====================================================

        HF_DEVICE_MAP = getattr(
            model,
            "hf_device_map",
            None
        )

        device_map_gpus = set()

        if HF_DEVICE_MAP:

            print(
                "Hugging Face device map:"
            )

            print("-" * 70)

            for (
                module_name,
                device
            ) in HF_DEVICE_MAP.items():

                print(
                    f"{module_name:55} -> "
                    f"{device}"
                )

                if isinstance(
                    device,
                    int
                ):

                    device_map_gpus.add(
                        device
                    )

                elif isinstance(
                    device,
                    str
                ):

                    if device.startswith(
                        "cuda:"
                    ):

                        try:

                            device_map_gpus.add(
                                int(
                                    device.split(
                                        ":"
                                    )[1]
                                )
                            )

                        except Exception:
                            pass

            print("-" * 70)


        # ====================================================
        # 12. REAL PARAMETER PLACEMENT
        # ====================================================

        parameter_gpus = set()

        for parameter in (
            model.parameters()
        ):

            if (
                parameter.device.type
                == "cuda"
            ):

                if (
                    parameter.device.index
                    is not None
                ):

                    parameter_gpus.add(
                        parameter.device.index
                    )


        verified_gpus = (
            device_map_gpus
            | parameter_gpus
        )


        MODEL_SHARDING_VERIFIED = (
            len(verified_gpus) >= 2
        )


        print()
        print("=" * 70)
        print(
            "MULTI-GPU VERIFICATION"
        )
        print("=" * 70)

        print(
            "Device-map GPUs:",
            sorted(
                device_map_gpus
            )
        )

        print(
            "Parameter GPUs:",
            sorted(
                parameter_gpus
            )
        )

        print(
            "Verified GPUs:",
            sorted(
                verified_gpus
            )
        )

        print(
            "Model sharding verified:",
            MODEL_SHARDING_VERIFIED
        )

        print("=" * 70)


        # ====================================================
        # 13. MEMORY
        # ====================================================

        GPU_MEMORY_AFTER_LOAD = {}

        for i in range(GPU_COUNT):

            allocated = (
                torch.cuda
                .memory_allocated(i)
                / (1024 ** 3)
            )

            reserved = (
                torch.cuda
                .memory_reserved(i)
                / (1024 ** 3)
            )

            free, total = (
                torch.cuda
                .mem_get_info(i)
            )

            GPU_MEMORY_AFTER_LOAD[i] = {

                "allocated_gb":
                    round(
                        allocated,
                        3
                    ),

                "reserved_gb":
                    round(
                        reserved,
                        3
                    ),

                "free_gb":
                    round(
                        free
                        / (1024 ** 3),
                        3
                    ),

                "total_gb":
                    round(
                        total
                        / (1024 ** 3),
                        3
                    )
            }


        print(
            "GPU memory after load:"
        )

        print(
            json.dumps(
                GPU_MEMORY_AFTER_LOAD,
                indent=2
            )
        )


        # ====================================================
        # 14. SIMPLE INFERENCE
        # ====================================================

        LLM_INFERENCE_VERIFIED = False

        try:

            test_prompt = (
                "Explain in two short sentences "
                "how MYRAA can use two GPUs "
                "for model sharding."
            )

            inputs = tokenizer(
                test_prompt,
                return_tensors="pt"
            )


            # First model device.
            try:

                input_device = (
                    model
                    .get_input_embeddings()
                    .weight
                    .device
                )

            except Exception:

                input_device = (
                    next(
                        model.parameters()
                    ).device
                )


            inputs = {
                key: value.to(
                    input_device
                )
                for key, value
                in inputs.items()
            }


            print()
            print(
                "Running inference..."
            )


            with torch.inference_mode():

                output = (
                    model.generate(
                        **inputs,
                        max_new_tokens=64,
                        do_sample=False,
                        pad_token_id=(
                            tokenizer.pad_token_id
                        ),
                        eos_token_id=(
                            tokenizer.eos_token_id
                        )
                    )
                )


            response = (
                tokenizer.decode(
                    output[0],
                    skip_special_tokens=True
                )
            )


            LLM_INFERENCE_VERIFIED = True

            print()
            print(
                "Inference output:"
            )

            print("-" * 70)

            print(
                response
            )

            print("-" * 70)


        except Exception as exc:

            print()
            print(
                "Inference failed:"
            )

            print(
                f"{type(exc).__name__}: "
                f"{exc}"
            )


        # ====================================================
        # 15. FINAL RESULT
        # ====================================================

        MYRAA_MULTI_GPU_RESULT = {

            "status":
                "success",

            "model":
                MODEL_ID,

            "configured_4bit":
                USE_4BIT,

            "actual_4bit":
                (
                    USE_4BIT
                    and
                    BITSANDBYTES_IMPORT_OK
                ),

            "bitsandbytes_available":
                BITSANDBYTES_AVAILABLE,

            "bitsandbytes_import_ok":
                BITSANDBYTES_IMPORT_OK,

            "bitsandbytes_version":
                BITSANDBYTES_VERSION,

            "dtype":
                TORCH_DTYPE_NAME,

            "gpu_count":
                GPU_COUNT,

            "device_map_gpus":
                sorted(
                    device_map_gpus
                ),

            "parameter_gpus":
                sorted(
                    parameter_gpus
                ),

            "verified_gpus":
                sorted(
                    verified_gpus
                ),

            "model_sharding_verified":
                MODEL_SHARDING_VERIFIED,

            "llm_inference_verified":
                LLM_INFERENCE_VERIFIED,

            "cpu_offload_enabled":
                False,

            "gpu_memory_after_load":
                GPU_MEMORY_AFTER_LOAD,

            "load_time_seconds":
                round(
                    LOAD_TIME,
                    3
                )
        }


        # ====================================================
        # 16. PUBLISH GLOBALS
        # ====================================================

        globals()[
            "MODEL_SHARDING_VERIFIED"
        ] = (
            MODEL_SHARDING_VERIFIED
        )

        globals()[
            "LLM_INFERENCE_VERIFIED"
        ] = (
            LLM_INFERENCE_VERIFIED
        )

        globals()[
            "MYRAA_MULTI_GPU_RESULT"
        ] = (
            MYRAA_MULTI_GPU_RESULT
        )

        globals()[
            "GPU_MEMORY_AFTER_LOAD"
        ] = (
            GPU_MEMORY_AFTER_LOAD
        )


        print()
        print("=" * 70)
        print(
            "MYRAA MODEL RESULT"
        )
        print("=" * 70)

        print(
            json.dumps(
                MYRAA_MULTI_GPU_RESULT,
                indent=2
            )
        )

        print("=" * 70)


# ============================================================
# 17. FINAL FLAGS
# ============================================================

if (
    "MODEL_SHARDING_VERIFIED"
    not in globals()
):

    MODEL_SHARDING_VERIFIED = False


if (
    "LLM_INFERENCE_VERIFIED"
    not in globals()
):

    LLM_INFERENCE_VERIFIED = False


globals()[
    "MODEL_SHARDING_VERIFIED"
] = (
    MODEL_SHARDING_VERIFIED
)

globals()[
    "LLM_INFERENCE_VERIFIED"
] = (
    LLM_INFERENCE_VERIFIED
)


print()
print("=" * 70)
print("CELL 7 FINAL STATUS")
print("=" * 70)

print(
    "bitsandbytes import OK :",
    globals().get(
        "BITSANDBYTES_IMPORT_OK",
        False
    )
)

print(
    "Model sharding         :",
    MODEL_SHARDING_VERIFIED
)

print(
    "LLM inference          :",
    LLM_INFERENCE_VERIFIED
)

print("=" * 70)

MYRAA — MODEL LOADER
CUDA available : True
GPU count      : 2
GPU 0: Tesla T4 | 14.56 GB | compute 7.5
GPU 1: Tesla T4 | 14.56 GB | compute 7.5
Model         : Qwen/Qwen2.5-32B-Instruct
4-bit config  : True
dtype         : float16
Checking bitsandbytes...
bitsandbytes module: False
4-BIT MODEL LOAD BLOCKED
The current Python kernel cannot successfully import bitsandbytes.

This is NOT a GPU/sharding failure.
Transformers requires bitsandbytes for 4-bit NF4 quantization.

Do NOT continue to model loading.
{
  "status": "blocked",
  "model": "Qwen/Qwen2.5-32B-Instruct",
  "configured_4bit": true,
  "actual_4bit": false,
  "bitsandbytes_available": false,
  "bitsandbytes_import_ok": false,
  "bitsandbytes_version": null,
  "bitsandbytes_error": null,
  "model_sharding_verified": false,
  "llm_inference_verified": false,
  "reason": "bitsandbytes cannot be imported by the current Python kernel."
}

CELL 7 FINAL STATUS
bitsandbytes import OK : False
Model sharding         : False
LLM infere

In [14]:
# ============================================================
# CELL 2 — DEPENDENCIES
# KAGGLE-SAFE VERSION
# ============================================================

import sys
import importlib.util


REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "safetensors": "safetensors",
    "sentencepiece": "sentencepiece",
}

OPTIONAL_PACKAGES = {
    "bitsandbytes": "bitsandbytes",
}


def check_package(import_name):

    return importlib.util.find_spec(
        import_name
    ) is not None


print("=" * 70)
print("MYRAA DEPENDENCY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# Required packages
# ------------------------------------------------------------

missing_required = []

for import_name, package_name in REQUIRED_PACKAGES.items():

    if check_package(import_name):

        print(
            f"✓ {package_name}"
        )

    else:

        missing_required.append(
            package_name
        )

        print(
            f"✗ {package_name}"
        )


# ------------------------------------------------------------
# Optional packages
# ------------------------------------------------------------

optional_available = {}

for import_name, package_name in OPTIONAL_PACKAGES.items():

    available = check_package(
        import_name
    )

    optional_available[package_name] = available

    if available:

        print(
            f"✓ optional: {package_name}"
        )

    else:

        print(
            f"⚠ optional: {package_name} "
            f"not installed"
        )


# ------------------------------------------------------------
# Do NOT automatically pip-install in the worker.
#
# Kaggle may have restricted/no network access.
# ------------------------------------------------------------

if missing_required:

    raise RuntimeError(
        "Required packages are missing: "
        + ", ".join(missing_required)
        + ". "
        "Install them using the Kaggle environment/package "
        "configuration before running the worker."
    )


BITSANDBYTES_AVAILABLE = (
    optional_available["bitsandbytes"]
)


print()
print(
    "bitsandbytes available:",
    BITSANDBYTES_AVAILABLE
)

print("=" * 70)
print("DEPENDENCY CHECK COMPLETE")
print("=" * 70)

MYRAA DEPENDENCY CHECK
✓ transformers
✓ accelerate
✓ safetensors
✓ sentencepiece
⚠ optional: bitsandbytes not installed

bitsandbytes available: False
DEPENDENCY CHECK COMPLETE


In [15]:
# ============================================================
# CELL 3 — HARDWARE DISCOVERY
# ============================================================

import torch
import platform
import subprocess
import json
import os

print("=" * 70)
print("MYRAA HARDWARE DISCOVERY")
print("=" * 70)

print("\nPyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

GPU_INFO = []

for i in range(torch.cuda.device_count()):

    props = torch.cuda.get_device_properties(i)

    total_vram_gb = props.total_memory / (1024 ** 3)

    info = {
        "index": i,
        "name": props.name,
        "vram_gb": round(total_vram_gb, 2),
        "compute_capability": f"{props.major}.{props.minor}",
    }

    GPU_INFO.append(info)

    print(f"\nGPU {i}")
    print(" Name:", props.name)
    print(" VRAM:", round(total_vram_gb, 2), "GB")
    print(
        " Compute capability:",
        f"{props.major}.{props.minor}"
    )

AGGREGATE_VRAM = sum(
    gpu["vram_gb"]
    for gpu in GPU_INFO
)

print("\nAggregate VRAM:",
      round(AGGREGATE_VRAM, 2),
      "GB")

print("\nCPU:")
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())

print("\nGPU status:")
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )

    print(result.stdout)

except Exception as e:
    print("nvidia-smi unavailable:", e)

MYRAA HARDWARE DISCOVERY

PyTorch: 2.11.0+cu128
CUDA: 12.8
CUDA available: True
GPU count: 2

GPU 0
 Name: Tesla T4
 VRAM: 14.56 GB
 Compute capability: 7.5

GPU 1
 Name: Tesla T4
 VRAM: 14.56 GB
 Compute capability: 7.5

Aggregate VRAM: 29.12 GB

CPU:
Platform: Linux-6.18.48+-x86_64-with-glibc2.39
CPU count: 4

GPU status:
Wed Oct  7 17:51:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================

In [16]:
# ============================================================
# CELL 4 — CUDA / FP16 VALIDATION
# ============================================================

import torch
import time

print("=" * 70)
print("CUDA / FP16 VALIDATION")
print("=" * 70)

assert torch.cuda.is_available(), "CUDA is not available."

assert torch.cuda.device_count() >= 2, (
    "MYRAA worker requires at least 2 GPUs."
)

for gpu_id in range(torch.cuda.device_count()):

    device = torch.device(f"cuda:{gpu_id}")

    print(f"\nTesting GPU {gpu_id}")

    torch.cuda.empty_cache()

    size = 4096

    a = torch.randn(
        size,
        size,
        device=device,
        dtype=torch.float16
    )

    b = torch.randn(
        size,
        size,
        device=device,
        dtype=torch.float16
    )

    # Warmup
    for _ in range(5):
        _ = a @ b

    torch.cuda.synchronize(device)

    start = time.perf_counter()

    iterations = 20

    for _ in range(iterations):
        result = a @ b

    torch.cuda.synchronize(device)

    elapsed = time.perf_counter() - start

    peak_memory = (
        torch.cuda.max_memory_allocated(device)
        / (1024 ** 3)
    )

    print(
        "Average:",
        round(elapsed / iterations, 6),
        "seconds"
    )

    print(
        "Peak VRAM:",
        round(peak_memory, 3),
        "GB"
    )

    del a
    del b
    del result

    torch.cuda.empty_cache()

print("\nFP16 validation PASSED.")

CUDA / FP16 VALIDATION

Testing GPU 0
Average: 0.007081 seconds
Peak VRAM: 6.359 GB

Testing GPU 1
Average: 0.006545 seconds
Peak VRAM: 8.095 GB

FP16 validation PASSED.


In [17]:
# ============================================================
# CELL 5 — NCCL MULTI-GPU COMMUNICATION
# KAGGLE / PYTHON 3.13 / TORCHRUN VERSION
# ============================================================

import os
import sys
import socket
import subprocess
import textwrap
import torch


print("=" * 70)
print("MYRAA — NCCL MULTI-GPU COMMUNICATION TEST")
print("=" * 70)

# ============================================================
# 1. BASIC GPU CHECK
# ============================================================

assert torch.cuda.is_available(), "CUDA is not available."

world_size = torch.cuda.device_count()

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {world_size}")

assert world_size >= 2, (
    f"MYRAA NCCL test requires at least 2 GPUs. Found {world_size}."
)

for i in range(world_size):

    props = torch.cuda.get_device_properties(i)

    print(
        f"GPU {i}: "
        f"{props.name} | "
        f"{props.total_memory / (1024 ** 3):.2f} GB"
    )

print("=" * 70)


# ============================================================
# 2. FIND FREE TCP PORT
# ============================================================

sock = socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM
)

sock.bind(("127.0.0.1", 0))

master_port = sock.getsockname()[1]

sock.close()

master_addr = "127.0.0.1"

print(f"MASTER_ADDR : {master_addr}")
print(f"MASTER_PORT : {master_port}")
print("=" * 70)


# ============================================================
# 3. CREATE REAL PYTHON WORKER FILE
#
# IMPORTANT:
# The worker is NOT defined inside the notebook.
# Therefore Python 3.13 does not need to pickle __main__.
# ============================================================

worker_path = "/tmp/myraa_nccl_worker.py"

worker_code = r'''
import os
import torch
import torch.distributed as dist


def main():

    # --------------------------------------------------------
    # torchrun supplies these variables automatically
    # --------------------------------------------------------

    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    # --------------------------------------------------------
    # Select this process's GPU
    # --------------------------------------------------------

    torch.cuda.set_device(local_rank)

    device = torch.device(
        f"cuda:{local_rank}"
    )

    print(
        f"[RANK {rank}] "
        f"LOCAL_RANK={local_rank} "
        f"GPU={torch.cuda.get_device_name(local_rank)}",
        flush=True
    )

    # --------------------------------------------------------
    # Initialize NCCL
    # --------------------------------------------------------

    dist.init_process_group(
        backend="nccl",
        init_method="env://",
    )

    print(
        f"[RANK {rank}] "
        f"NCCL process group initialized",
        flush=True
    )

    # --------------------------------------------------------
    # Each GPU starts with a unique value
    #
    # GPU 0 -> 1
    # GPU 1 -> 2
    #
    # Expected result:
    #
    # 1 + 2 = 3
    # --------------------------------------------------------

    tensor = torch.tensor(
        [float(rank + 1)],
        dtype=torch.float32,
        device=device,
    )

    print(
        f"[RANK {rank}] "
        f"BEFORE all_reduce: {tensor.item()}",
        flush=True
    )

    # --------------------------------------------------------
    # REAL NCCL GPU-TO-GPU COMMUNICATION
    # --------------------------------------------------------

    dist.all_reduce(
        tensor,
        op=dist.ReduceOp.SUM,
    )

    torch.cuda.synchronize(device)

    print(
        f"[RANK {rank}] "
        f"AFTER all_reduce: {tensor.item()}",
        flush=True
    )

    # --------------------------------------------------------
    # Validate
    # --------------------------------------------------------

    expected = float(
        world_size * (world_size + 1) / 2
    )

    actual = tensor.item()

    if abs(actual - expected) > 0.0001:

        raise RuntimeError(
            f"[RANK {rank}] "
            f"NCCL RESULT FAILED: "
            f"expected {expected}, "
            f"got {actual}"
        )

    print(
        f"[RANK {rank}] "
        f"all_reduce result VERIFIED",
        flush=True
    )

    # --------------------------------------------------------
    # Synchronize all ranks
    # --------------------------------------------------------

    dist.barrier()

    print(
        f"[RANK {rank}] "
        f"NCCL communication PASSED",
        flush=True
    )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    dist.destroy_process_group()


if __name__ == "__main__":
    main()
'''

with open(worker_path, "w") as f:
    f.write(textwrap.dedent(worker_code))

print(f"Worker created: {worker_path}")


# ============================================================
# 4. VERIFY WORKER FILE EXISTS
# ============================================================

assert os.path.exists(worker_path)

print("Worker file verified.")


# ============================================================
# 5. LAUNCH REAL DISTRIBUTED JOB WITH TORCHRUN
# ============================================================

torchrun = os.path.join(
    os.path.dirname(sys.executable),
    "torchrun"
)

if not os.path.exists(torchrun):

    # Fallback for environments where torchrun is not
    # installed beside the Python executable.
    torchrun = "torchrun"

print(f"torchrun: {torchrun}")

print("=" * 70)
print("LAUNCHING NCCL WORKERS")
print("=" * 70)


env = os.environ.copy()

env["MASTER_ADDR"] = master_addr
env["MASTER_PORT"] = str(master_port)

# Useful for cloud GPU environments.
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"

# Do not force a network interface here.
# NCCL will choose the appropriate local transport.


command = [
    torchrun,

    "--standalone",

    f"--nproc_per_node={world_size}",

    worker_path,
]


print("Command:")
print(" ".join(command))
print("=" * 70)


result = subprocess.run(
    command,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


# ============================================================
# 6. SHOW COMPLETE TORCHRUN OUTPUT
# ============================================================

print(result.stdout)


# ============================================================
# 7. VALIDATE RESULT
# ============================================================

if result.returncode != 0:

    print("=" * 70)
    print("NCCL MULTI-GPU COMMUNICATION FAILED")
    print("=" * 70)

    raise RuntimeError(
        f"torchrun exited with code {result.returncode}."
    )


print("=" * 70)
print("NCCL MULTI-GPU COMMUNICATION PASSED")
print("=" * 70)

print()
print("MYRAA VERIFIED:")
print("  ✓ CUDA available")
print("  ✓ 2 GPUs detected")
print("  ✓ torchrun launched one process per GPU")
print("  ✓ NCCL process group initialized")
print("  ✓ GPU-to-GPU all_reduce executed")
print("  ✓ Reduction result was correct")
print("  ✓ All ranks synchronized")
print("  ✓ NCCL multi-GPU communication verified")
print("=" * 70)

MYRAA — NCCL MULTI-GPU COMMUNICATION TEST
CUDA available : True
GPU count      : 2
GPU 0: Tesla T4 | 14.56 GB
GPU 1: Tesla T4 | 14.56 GB
MASTER_ADDR : 127.0.0.1
MASTER_PORT : 57819
Worker created: /tmp/myraa_nccl_worker.py
Worker file verified.
torchrun: torchrun
LAUNCHING NCCL WORKERS
Command:
torchrun --standalone --nproc_per_node=2 /tmp/myraa_nccl_worker.py
W1007 17:51:15.315000 265 torch/distributed/run.py:851] 
W1007 17:51:15.315000 265 torch/distributed/run.py:851] *****************************************
W1007 17:51:15.315000 265 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1007 17:51:15.315000 265 torch/distributed/run.py:851] *****************************************
[RANK 0] LOCAL_RANK=0 GPU=Tesla T4
[RANK 1] LOCAL_RANK=1 GPU=Tesla T4
[W1007 17:51:17.941308634 Utils.hpp:137] Warning

In [20]:
# ============================================================
# CELL 6 — MYRAA CAPABILITY REPORT
# JSON-SAFE / TRUTHFUL / STATE-AWARE
# ============================================================

import json
import torch
from pathlib import Path


print("=" * 70)
print("MYRAA CAPABILITY REPORT")
print("=" * 70)


# ============================================================
# 1. SAFE JSON CONVERTER
# ============================================================

def make_json_safe(value):

    if isinstance(value, torch.dtype):
        return str(value).replace("torch.", "")

    if isinstance(value, torch.device):
        return str(value)

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_safe(v)
            for v in value
        ]

    if isinstance(value, set):
        return [
            make_json_safe(v)
            for v in value
        ]

    if hasattr(value, "item"):

        try:
            return value.item()
        except Exception:
            pass

    if value is None:
        return None

    if isinstance(
        value,
        (str, int, float, bool)
    ):
        return value

    return str(value)


# ============================================================
# 2. PATH
# ============================================================

if "BASE_DIR" in globals():

    BASE_DIR = Path(
        BASE_DIR
    )

else:

    BASE_DIR = Path(
        "/kaggle/working/myraa_worker"
    )


BASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CAPABILITY_FILE = (
    BASE_DIR / "capabilities.json"
)


# ============================================================
# 3. WORKER CONFIG
# ============================================================

PROVIDER = globals().get(
    "PROVIDER",
    "kaggle"
)

WORKER_NAME = globals().get(
    "WORKER_NAME",
    "myraa-kaggle-t4x2"
)

WORKER_VERSION = globals().get(
    "WORKER_VERSION",
    "1.0.0"
)


# ============================================================
# 4. MODEL CONFIG
# ============================================================

MODEL_ID = globals().get(
    "MODEL_ID",
    "unknown"
)

USE_4BIT = bool(
    globals().get(
        "USE_4BIT",
        False
    )
)


TORCH_DTYPE_RAW = globals().get(
    "TORCH_DTYPE",
    "float16"
)


if isinstance(
    TORCH_DTYPE_RAW,
    torch.dtype
):

    TORCH_DTYPE_NAME = (
        str(TORCH_DTYPE_RAW)
        .replace("torch.", "")
    )

else:

    TORCH_DTYPE_NAME = str(
        TORCH_DTYPE_RAW
    )


# ============================================================
# 5. CUDA / GPU
# ============================================================

CUDA_AVAILABLE = bool(
    torch.cuda.is_available()
)

GPU_COUNT = (
    torch.cuda.device_count()
    if CUDA_AVAILABLE
    else 0
)


# ============================================================
# 6. GPU INFORMATION
# ============================================================

if "GPU_INFO" in globals():

    GPU_INFO = make_json_safe(
        globals()["GPU_INFO"]
    )

else:

    GPU_INFO = []

    if CUDA_AVAILABLE:

        for i in range(
            GPU_COUNT
        ):

            props = (
                torch.cuda
                .get_device_properties(i)
            )

            total_gb = (
                props.total_memory
                / (1024 ** 3)
            )

            GPU_INFO.append({

                "index": i,

                "name": str(
                    props.name
                ),

                "vram_gb": round(
                    total_gb,
                    2
                ),

                "compute_capability": (
                    f"{props.major}."
                    f"{props.minor}"
                )
            })


# ============================================================
# 7. AGGREGATE VRAM
# ============================================================

AGGREGATE_VRAM = round(
    sum(
        float(
            gpu.get(
                "vram_gb",
                0
            )
        )
        for gpu in GPU_INFO
    ),
    2
)


# ============================================================
# 8. NCCL
# ============================================================

NCCL_AVAILABLE = bool(
    torch.distributed.is_nccl_available()
)


# ============================================================
# 9. VERIFIED TEST RESULTS
#
# IMPORTANT:
# These are taken from the CURRENT notebook globals.
# We do not invent successful capabilities.
# ============================================================

NCCL_COMMUNICATION_VERIFIED = bool(
    globals().get(
        "NCCL_COMMUNICATION_VERIFIED",
        False
    )
)


MODEL_SHARDING_VERIFIED = bool(
    globals().get(
        "MODEL_SHARDING_VERIFIED",
        False
    )
)


LLM_INFERENCE_VERIFIED = bool(
    globals().get(
        "LLM_INFERENCE_VERIFIED",
        False
    )
)


TRAINING_VERIFIED = bool(
    globals().get(
        "TRAINING_VERIFIED",
        False
    )
)


EMBEDDINGS_VERIFIED = bool(
    globals().get(
        "EMBEDDINGS_VERIFIED",
        False
    )
)


# ============================================================
# 10. DERIVED CAPABILITIES
# ============================================================

FP16_AVAILABLE = bool(
    CUDA_AVAILABLE
)

MULTI_GPU_CUDA_AVAILABLE = bool(
    CUDA_AVAILABLE
    and GPU_COUNT >= 2
)


# ============================================================
# 11. BUILD REPORT
# ============================================================

CAPABILITIES = {

    "schema_version": "1.1",

    "provider": PROVIDER,

    "worker_name": WORKER_NAME,

    "worker_version": WORKER_VERSION,

    "pytorch": str(
        torch.__version__
    ),

    "cuda": (
        str(torch.version.cuda)
        if torch.version.cuda is not None
        else None
    ),

    "cuda_available": CUDA_AVAILABLE,

    "gpu_count": GPU_COUNT,

    "gpus": GPU_INFO,

    "aggregate_vram_gb": AGGREGATE_VRAM,

    "features": {

        "fp16": FP16_AVAILABLE,

        "nccl_installed": NCCL_AVAILABLE,

        "multi_gpu_cuda": (
            MULTI_GPU_CUDA_AVAILABLE
        ),

        "nccl_communication": (
            NCCL_COMMUNICATION_VERIFIED
        ),

        "model_sharding": (
            MODEL_SHARDING_VERIFIED
        ),

        "llm_inference": (
            LLM_INFERENCE_VERIFIED
        ),

        "training": (
            TRAINING_VERIFIED
        ),

        "embeddings": (
            EMBEDDINGS_VERIFIED
        )
    },

    "model": {

        "configured_model": str(
            MODEL_ID
        ),

        "quantization": (
            "4bit"
            if USE_4BIT
            else "none"
        ),

        "dtype": TORCH_DTYPE_NAME
    }
}


# ============================================================
# 12. FINAL JSON-SAFETY PASS
# ============================================================

CAPABILITIES = make_json_safe(
    CAPABILITIES
)


# ============================================================
# 13. SERIALIZATION TEST
# ============================================================

CAPABILITY_JSON = json.dumps(
    CAPABILITIES,
    indent=2,
    ensure_ascii=False
)


# ============================================================
# 14. SAVE
# ============================================================

with open(
    CAPABILITY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        CAPABILITY_JSON
    )


# ============================================================
# 15. DISPLAY
# ============================================================

print(
    CAPABILITY_JSON
)

print()
print(
    "Capability report saved to:"
)

print(
    CAPABILITY_FILE
)

print("=" * 70)
print("CELL 6 COMPLETE")
print("=" * 70)

MYRAA CAPABILITY REPORT
{
  "schema_version": "1.1",
  "provider": "kaggle",
  "worker_name": "myraa-kaggle-t4x2",
  "worker_version": "1.0.0",
  "pytorch": "2.11.0+cu128",
  "cuda": "12.8",
  "cuda_available": true,
  "gpu_count": 2,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "vram_gb": 14.56,
      "compute_capability": "7.5"
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "vram_gb": 14.56,
      "compute_capability": "7.5"
    }
  ],
  "aggregate_vram_gb": 29.12,
  "features": {
    "fp16": true,
    "nccl_installed": true,
    "multi_gpu_cuda": true,
    "nccl_communication": false,
    "model_sharding": true,
    "llm_inference": false,
    "training": false,
    "embeddings": false
  },
  "model": {
    "configured_model": "Qwen/Qwen2.5-32B-Instruct",
    "quantization": "none",
    "dtype": "float16"
  }
}

Capability report saved to:
/kaggle/working/myraa_worker/capabilities.json
CELL 6 COMPLETE


In [19]:
# ======================================================================
# MYRAA — CELL 7
# MULTI-GPU MODEL SHARDING + INFERENCE TEST
# ======================================================================

import os
import gc
import json
import time
import traceback

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# ======================================================================
# CONFIGURATION
# ======================================================================

# IMPORTANT:
# Never allow "unknown" to become a Hugging Face model ID.
DEFAULT_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

MODEL_ID = (
    os.getenv("MYRAA_MODEL_ID")
    or globals().get("MODEL_ID")
    or DEFAULT_MODEL_ID
)

if not isinstance(MODEL_ID, str) or not MODEL_ID.strip():
    MODEL_ID = DEFAULT_MODEL_ID

MODEL_ID = MODEL_ID.strip()

if MODEL_ID.lower() == "unknown":
    MODEL_ID = DEFAULT_MODEL_ID

# For the 7B verification test, keep this False.
# Once the basic multi-GPU path is confirmed, you can switch to 4-bit
# for larger models if bitsandbytes is actually available.
USE_4BIT = bool(
    os.getenv("MYRAA_USE_4BIT", "").lower() in
    ("1", "true", "yes", "on")
)

# Safe default for the T4 GPUs.
TORCH_DTYPE = torch.float16

MAX_NEW_TOKENS = int(os.getenv("MYRAA_MAX_NEW_TOKENS", "80"))

# Keep some VRAM free for CUDA/runtime overhead.
GPU_MEMORY_LIMIT = "13GiB"

# Do NOT put quantized modules on CPU/disk in this test.
DEVICE_MAP = "auto"


# ======================================================================
# HEADER
# ======================================================================

print("=" * 70)
print("MYRAA — MULTI-GPU MODEL SHARDING TEST")
print("=" * 70)

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {torch.cuda.device_count()}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

if torch.cuda.device_count() < 2:
    raise RuntimeError(
        f"MYRAA requires 2 GPUs for this test. "
        f"Detected: {torch.cuda.device_count()}"
    )

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    total_gb = props.total_memory / (1024 ** 3)

    print(
        f"GPU {i}: {props.name} | "
        f"{total_gb:.2f} GB | "
        f"compute {props.major}.{props.minor}"
    )

print("=" * 70)
print(f"Model        : {MODEL_ID}")
print(f"4-bit        : {USE_4BIT}")
print(f"dtype        : {str(TORCH_DTYPE).replace('torch.', '')}")
print("=" * 70)


# ======================================================================
# VERIFY MODEL CONFIGURATION
# ======================================================================

if MODEL_ID.lower() in ("unknown", "none", "null"):
    raise RuntimeError(
        "Invalid MODEL_ID detected. "
        "Set MYRAA_MODEL_ID or use the built-in default."
    )

print(f"Validated model ID: {MODEL_ID}")


# ======================================================================
# CHECK OPTIONAL 4-BIT DEPENDENCY
# ======================================================================

BITSANDBYTES_AVAILABLE = False

if USE_4BIT:
    try:
        import bitsandbytes as bnb

        BITSANDBYTES_AVAILABLE = True

        print("=" * 70)
        print("bitsandbytes : AVAILABLE")
        print(f"bitsandbytes version: {getattr(bnb, '__version__', 'unknown')}")
        print("=" * 70)

    except Exception as e:
        print("=" * 70)
        print("bitsandbytes : NOT AVAILABLE")
        print(f"Reason: {type(e).__name__}: {e}")
        print("=" * 70)

        print("Falling back to FP16 for this Cell 7 test.")
        USE_4BIT = False

else:
    print("=" * 70)
    print("Actual 4-bit : False")
    print("Mode         : FP16 multi-GPU sharding")
    print("=" * 70)


# ======================================================================
# CLEAN GPU MEMORY
# ======================================================================

print("=" * 70)
print("Cleaning GPU memory...")
print("=" * 70)

try:
    if "model" in globals():
        del model
except Exception:
    pass

try:
    if "tokenizer" in globals():
        del tokenizer
except Exception:
    pass

gc.collect()

for i in range(torch.cuda.device_count()):
    try:
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

print("GPU memory cleanup complete.")


# ======================================================================
# GPU MEMORY BEFORE LOADING
# ======================================================================

print("=" * 70)
print("GPU memory BEFORE model loading:")
print("-" * 70)

for i in range(torch.cuda.device_count()):
    free_mem, total_mem = torch.cuda.mem_get_info(i)

    print(
        f"GPU {i}: "
        f"{free_mem / (1024 ** 3):.2f} GB free / "
        f"{total_mem / (1024 ** 3):.2f} GB total"
    )

print("-" * 70)


# ======================================================================
# TOKENIZER
# ======================================================================

print("=" * 70)
print(f"Initializing tokenizer for model: {MODEL_ID}")
print("=" * 70)

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Tokenizer initialized successfully.")

except Exception as e:
    print("=" * 70)
    print("TOKENIZER INITIALIZATION FAILED")
    print("=" * 70)
    print(f"Model ID: {MODEL_ID}")
    print(f"{type(e).__name__}: {e}")
    print("=" * 70)

    # Publish failure state for later cells.
    MODEL_SHARDING_VERIFIED = False
    LLM_INFERENCE_VERIFIED = False

    MYRAA_MULTI_GPU_RESULT = {
        "success": False,
        "stage": "tokenizer",
        "model_id": MODEL_ID,
        "error_type": type(e).__name__,
        "error": str(e),
    }

    raise


# ======================================================================
# MODEL LOAD CONFIG
# ======================================================================

load_kwargs = {
    "device_map": DEVICE_MAP,
    "torch_dtype": TORCH_DTYPE,
    "low_cpu_mem_usage": True,
    "trust_remote_code": True,
    "max_memory": {
        0: GPU_MEMORY_LIMIT,
        1: GPU_MEMORY_LIMIT,
    },
}

if USE_4BIT and BITSANDBYTES_AVAILABLE:
    load_kwargs.update({
        "load_in_4bit": True,
        "bnb_4bit_compute_dtype": TORCH_DTYPE,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_use_double_quant": True,
    })


# ======================================================================
# MODEL LOAD
# ======================================================================

print("=" * 70)
print("Loading model across GPUs...")
print("=" * 70)

print(f"device_map   : {DEVICE_MAP}")
print(f"max GPU mem  : {GPU_MEMORY_LIMIT} per GPU")
print(
    f"quantization : "
    f"{'4-bit' if USE_4BIT else 'FP16'}"
)

load_start = time.time()

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        **load_kwargs,
    )

    model.eval()

    load_time = time.time() - load_start

    print("=" * 70)
    print("MODEL LOAD SUCCESSFUL")
    print(f"Load time: {load_time:.2f} seconds")
    print("=" * 70)

except Exception as e:
    load_time = time.time() - load_start

    print("=" * 70)
    print("MODEL LOAD FAILED")
    print("=" * 70)
    print(f"Model: {MODEL_ID}")
    print(f"Time : {load_time:.2f} seconds")
    print(f"{type(e).__name__}: {e}")
    print("=" * 70)

    MODEL_SHARDING_VERIFIED = False
    LLM_INFERENCE_VERIFIED = False

    MYRAA_MULTI_GPU_RESULT = {
        "success": False,
        "stage": "model_load",
        "model_id": MODEL_ID,
        "error_type": type(e).__name__,
        "error": str(e),
        "load_time_seconds": round(load_time, 3),
    }

    raise


# ======================================================================
# DEVICE MAP INSPECTION
# ======================================================================

print("=" * 70)
print("MODEL DEVICE MAP")
print("=" * 70)

hf_device_map = getattr(model, "hf_device_map", None)

if hf_device_map is not None:
    for module_name, device in hf_device_map.items():
        print(f"{module_name:50s} -> {device}")
else:
    print("No hf_device_map exposed by this model.")


# ======================================================================
# ACTUAL PARAMETER PLACEMENT
# ======================================================================

def normalize_cuda_device(device):
    """
    Convert Hugging Face / torch device representations into:
        0
        1
        None
    """
    if isinstance(device, int):
        return device

    if isinstance(device, torch.device):
        if device.type == "cuda":
            return device.index if device.index is not None else 0
        return None

    if isinstance(device, str):
        value = device.lower().strip()

        if value.startswith("cuda:"):
            try:
                return int(value.split(":")[1])
            except Exception:
                return None

        if value.isdigit():
            return int(value)

    return None


used_gpu_ids = set()
parameter_count = 0

for name, parameter in model.named_parameters():
    parameter_count += parameter.numel()

    gpu_id = normalize_cuda_device(parameter.device)

    if gpu_id is not None:
        used_gpu_ids.add(gpu_id)


# Also inspect hf_device_map.
if isinstance(hf_device_map, dict):
    for device in hf_device_map.values():
        gpu_id = normalize_cuda_device(device)

        if gpu_id is not None:
            used_gpu_ids.add(gpu_id)


print("=" * 70)
print("ACTUAL MODEL GPU PLACEMENT")
print("=" * 70)

print(f"GPUs containing model modules: {sorted(used_gpu_ids)}")
print(f"Parameter count: {parameter_count:,}")

MODEL_SHARDING_VERIFIED = (
    len(used_gpu_ids) >= 2
)

print(
    f"MODEL_SHARDING_VERIFIED: "
    f"{MODEL_SHARDING_VERIFIED}"
)


# ======================================================================
# GPU MEMORY AFTER LOAD
# ======================================================================

print("=" * 70)
print("GPU memory AFTER model loading:")
print("-" * 70)

gpu_memory_after_load = {}

for i in range(torch.cuda.device_count()):
    free_mem, total_mem = torch.cuda.mem_get_info(i)

    allocated = torch.cuda.memory_allocated(i) / (1024 ** 3)
    reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)

    gpu_memory_after_load[str(i)] = {
        "free_gb": round(free_mem / (1024 ** 3), 3),
        "total_gb": round(total_mem / (1024 ** 3), 3),
        "allocated_gb": round(allocated, 3),
        "reserved_gb": round(reserved, 3),
    }

    print(
        f"GPU {i}: "
        f"{free_mem / (1024 ** 3):.2f} GB free / "
        f"{total_mem / (1024 ** 3):.2f} GB total | "
        f"allocated {allocated:.2f} GB | "
        f"reserved {reserved:.2f} GB"
    )

print("-" * 70)


# ======================================================================
# REAL MULTI-GPU INFERENCE TEST
# ======================================================================

print("=" * 70)
print("Running real LLM inference...")
print("=" * 70)

prompt = (
    "You are MYRAA, a helpful Windows AI companion. "
    "Briefly explain why using two GPUs can help run a model "
    "that is too large for one GPU."
)

try:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    # Put input tensors on the device containing the model's input
    # embeddings. With device_map="auto", this is normally the first GPU.
    input_device = None

    if isinstance(hf_device_map, dict):
        for module_name, device in hf_device_map.items():
            if (
                "embed_tokens" in module_name
                or module_name in ("model.embed_tokens", "transformer.wte")
            ):
                gpu_id = normalize_cuda_device(device)

                if gpu_id is not None:
                    input_device = torch.device(f"cuda:{gpu_id}")
                    break

    if input_device is None:
        input_device = torch.device("cuda:0")

    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }

    inference_start = time.time()

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    inference_time = time.time() - inference_start

    generated_text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    )

    LLM_INFERENCE_VERIFIED = True

    print("=" * 70)
    print("LLM INFERENCE SUCCESSFUL")
    print(f"Inference time: {inference_time:.2f} seconds")
    print("-" * 70)
    print(generated_text)
    print("=" * 70)

except Exception as e:
    inference_time = None

    LLM_INFERENCE_VERIFIED = False

    print("=" * 70)
    print("LLM INFERENCE FAILED")
    print("=" * 70)
    print(f"{type(e).__name__}: {e}")
    print("=" * 70)

    generated_text = None


# ======================================================================
# FINAL RESULT
# ======================================================================

MYRAA_MULTI_GPU_RESULT = {
    "success": bool(
        MODEL_SHARDING_VERIFIED and
        LLM_INFERENCE_VERIFIED
    ),
    "model_id": MODEL_ID,
    "quantization": "4bit" if USE_4BIT else "fp16",
    "dtype": str(TORCH_DTYPE).replace("torch.", ""),
    "gpu_count": torch.cuda.device_count(),
    "gpus_used_for_model": sorted(used_gpu_ids),
    "parameter_count": int(parameter_count),
    "model_sharding_verified": bool(MODEL_SHARDING_VERIFIED),
    "llm_inference_verified": bool(LLM_INFERENCE_VERIFIED),
    "load_time_seconds": round(load_time, 3),
    "inference_time_seconds": (
        round(inference_time, 3)
        if inference_time is not None
        else None
    ),
    "gpu_memory_after_load": gpu_memory_after_load,
    "generated_text": generated_text,
}


# ======================================================================
# PUBLISH GLOBAL FLAGS
# ======================================================================

globals()["MODEL_SHARDING_VERIFIED"] = bool(
    MODEL_SHARDING_VERIFIED
)

globals()["LLM_INFERENCE_VERIFIED"] = bool(
    LLM_INFERENCE_VERIFIED
)

globals()["MYRAA_MULTI_GPU_RESULT"] = MYRAA_MULTI_GPU_RESULT

globals()["MYRAA_MODEL_ID"] = MODEL_ID
globals()["MYRAA_MODEL_QUANTIZATION"] = (
    "4bit" if USE_4BIT else "fp16"
)


# ======================================================================
# FINAL STATUS
# ======================================================================

print()
print("=" * 70)
print("MYRAA CELL 7 FINAL STATUS")
print("=" * 70)

print(f"Model ID                 : {MODEL_ID}")
print(f"Quantization             : {'4-bit' if USE_4BIT else 'FP16'}")
print(f"GPUs used by model       : {sorted(used_gpu_ids)}")
print(
    f"MODEL_SHARDING_VERIFIED  : "
    f"{MODEL_SHARDING_VERIFIED}"
)
print(
    f"LLM_INFERENCE_VERIFIED   : "
    f"{LLM_INFERENCE_VERIFIED}"
)
print(
    f"MYRAA_MULTI_GPU_RESULT   : "
    f"{MYRAA_MULTI_GPU_RESULT['success']}"
)

print("=" * 70)

MYRAA — MULTI-GPU MODEL SHARDING TEST
CUDA available : True
GPU count      : 2
GPU 0: Tesla T4 | 14.56 GB | compute 7.5
GPU 1: Tesla T4 | 14.56 GB | compute 7.5
Model        : Qwen/Qwen2.5-32B-Instruct
4-bit        : False
dtype        : float16
Validated model ID: Qwen/Qwen2.5-32B-Instruct
Actual 4-bit : False
Mode         : FP16 multi-GPU sharding
Cleaning GPU memory...
GPU memory cleanup complete.
GPU memory BEFORE model loading:
----------------------------------------------------------------------
GPU 0: 14.41 GB free / 14.56 GB total
GPU 1: 6.45 GB free / 14.56 GB total
----------------------------------------------------------------------
Initializing tokenizer for model: Qwen/Qwen2.5-32B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


MODEL LOAD SUCCESSFUL
Load time: 549.09 seconds
MODEL DEVICE MAP
model.embed_tokens                                 -> 0
model.layers.0                                     -> 0
model.layers.1                                     -> 0
model.layers.2                                     -> 0
model.layers.3                                     -> 0
model.layers.4                                     -> 0
model.layers.5                                     -> 0
model.layers.6                                     -> 0
model.layers.7                                     -> 0
model.layers.8                                     -> 0
model.layers.9                                     -> 0
model.layers.10                                    -> 0
model.layers.11                                    -> 1
model.layers.12                                    -> 1
model.layers.13                                    -> 1
model.layers.14                                    -> 1
model.layers.15                        

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 8 — MYRAA INFERENCE
# JSON-SAFE / DEPENDENCY-AWARE
# ============================================================

import json
import time
import torch


print("=" * 70)
print("MYRAA INFERENCE")
print("=" * 70)


# ============================================================
# CONFIGURATION
# ============================================================

MAX_NEW_TOKENS = int(
    globals().get(
        "MAX_NEW_TOKENS",
        256
    )
)

MAX_OUTPUT_TOKENS = int(
    globals().get(
        "MAX_OUTPUT_TOKENS",
        512
    )
)

TEMPERATURE = float(
    globals().get(
        "TEMPERATURE",
        0.7
    )
)

TOP_P = float(
    globals().get(
        "TOP_P",
        0.9
    )
)

MAX_INPUT_CHARS = int(
    globals().get(
        "MAX_INPUT_CHARS",
        12000
    )
)

DO_SAMPLE = bool(
    globals().get(
        "DO_SAMPLE",
        True
    )
)


# ============================================================
# MODEL STATUS
# ============================================================

MODEL_LOADED = (
    "model" in globals()
    and globals().get(
        "model"
    ) is not None
)


TOKENIZER_LOADED = (
    "tokenizer" in globals()
    and globals().get(
        "tokenizer"
    ) is not None
)


# ============================================================
# BLOCK IF MODEL IS MISSING
# ============================================================

if not MODEL_LOADED:

    INFERENCE_AVAILABLE = False

    LLM_INFERENCE_VERIFIED = False

    MYRAA_LAST_INFERENCE = {

        "status":
            "blocked",

        "model_loaded":
            False,

        "tokenizer_loaded":
            TOKENIZER_LOADED,

        "reason":
            "Model is not loaded. "
            "Run a successful Cell 7 first."
    }


    globals()[
        "INFERENCE_AVAILABLE"
    ] = INFERENCE_AVAILABLE

    globals()[
        "LLM_INFERENCE_VERIFIED"
    ] = False

    globals()[
        "MYRAA_LAST_INFERENCE"
    ] = MYRAA_LAST_INFERENCE


    print(
        "Inference blocked: "
        "model is not loaded."
    )

else:

    if not TOKENIZER_LOADED:

        raise RuntimeError(
            "Model is loaded but "
            "tokenizer is missing."
        )


    # ========================================================
    # VALIDATION
    # ========================================================

    if MAX_NEW_TOKENS < 1:
        raise ValueError(
            "MAX_NEW_TOKENS must be >= 1."
        )

    if MAX_OUTPUT_TOKENS < 1:
        raise ValueError(
            "MAX_OUTPUT_TOKENS must be >= 1."
        )

    if MAX_INPUT_CHARS < 1:
        raise ValueError(
            "MAX_INPUT_CHARS must be >= 1."
        )

    if TEMPERATURE < 0:
        raise ValueError(
            "TEMPERATURE cannot be negative."
        )

    if not 0.0 <= TOP_P <= 1.0:
        raise ValueError(
            "TOP_P must be between 0 and 1."
        )


    # ========================================================
    # INPUT DEVICE
    # ========================================================

    def get_model_input_device(
        loaded_model
    ):

        try:

            embedding = (
                loaded_model
                .get_input_embeddings()
            )

            if embedding is not None:

                return (
                    embedding
                    .weight
                    .device
                )

        except Exception:
            pass


        if hasattr(
            loaded_model,
            "hf_device_map"
        ):

            try:

                for device in (
                    loaded_model
                    .hf_device_map
                    .values()
                ):

                    if isinstance(
                        device,
                        int
                    ):

                        return torch.device(
                            f"cuda:{device}"
                        )


                    if isinstance(
                        device,
                        str
                    ):

                        value = (
                            device
                            .strip()
                            .lower()
                        )


                        if value.startswith(
                            "cuda:"
                        ):

                            return torch.device(
                                value
                            )


                        if value == "cpu":

                            return torch.device(
                                "cpu"
                            )

            except Exception:
                pass


        try:

            return next(
                loaded_model
                .parameters()
            ).device

        except StopIteration:

            return torch.device(
                "cuda:0"
                if torch.cuda.is_available()
                else "cpu"
            )


    # ========================================================
    # GENERATION
    # ========================================================

    def generate_response(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P
    ):

        if not isinstance(
            prompt,
            str
        ):

            raise TypeError(
                "prompt must be a string."
            )


        prompt = prompt.strip()


        if not prompt:

            raise ValueError(
                "prompt cannot be empty."
            )


        if len(prompt) > MAX_INPUT_CHARS:

            raise ValueError(
                f"Prompt exceeds "
                f"{MAX_INPUT_CHARS} characters."
            )


        generation_limit = min(
            int(max_new_tokens),
            MAX_OUTPUT_TOKENS
        )


        messages = [

            {
                "role": "system",
                "content": (
                    "You are MYRAA, a private AI "
                    "compute worker. Answer clearly "
                    "and accurately."
                )
            },

            {
                "role": "user",
                "content": prompt
            }
        ]


        try:

            text = (
                tokenizer
                .apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            )

        except Exception:

            text = (
                "System: You are MYRAA.\n\n"
                f"User: {prompt}\n\n"
                "Assistant:"
            )


        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True
        )


        input_device = (
            get_model_input_device(
                model
            )
        )


        inputs = {
            key: value.to(
                input_device
            )
            for key, value
            in inputs.items()
        }


        input_tokens = int(
            inputs[
                "input_ids"
            ].shape[-1]
        )


        generation_kwargs = {

            "max_new_tokens":
                generation_limit,

            "pad_token_id":
                tokenizer.pad_token_id,

            "eos_token_id":
                tokenizer.eos_token_id
        }


        if DO_SAMPLE:

            generation_kwargs.update({

                "do_sample":
                    True,

                "temperature":
                    float(temperature),

                "top_p":
                    float(top_p)
            })

        else:

            generation_kwargs[
                "do_sample"
            ] = False


        start = time.perf_counter()


        with torch.inference_mode():

            output = (
                model.generate(
                    **inputs,
                    **generation_kwargs
                )
            )


        elapsed = (
            time.perf_counter()
            - start
        )


        total_tokens = int(
            output.shape[-1]
        )


        generated_tokens = max(
            0,
            total_tokens
            - input_tokens
        )


        response = (
            tokenizer.decode(
                output[0][input_tokens:],
                skip_special_tokens=True
            )
            .strip()
        )


        tokens_per_second = (
            generated_tokens / elapsed
            if elapsed > 0
            else 0.0
        )


        return {

            "status":
                "success",

            "text":
                response,

            "input_tokens":
                int(input_tokens),

            "output_tokens":
                int(generated_tokens),

            "runtime_seconds":
                round(
                    elapsed,
                    3
                ),

            "tokens_per_second":
                round(
                    tokens_per_second,
                    2
                ),

            "device":
                str(input_device)
        }


    # ========================================================
    # TEST
    # ========================================================

    TEST_PROMPT = (
        "Explain how MYRAA should decide "
        "whether to use local CPU, local GPU, "
        "cloud AI, or a Kaggle T4x2 GPU worker."
    )


    try:

        result = generate_response(
            TEST_PROMPT
        )


        INFERENCE_AVAILABLE = True

        LLM_INFERENCE_VERIFIED = True

        MYRAA_LAST_INFERENCE = result


        globals()[
            "INFERENCE_AVAILABLE"
        ] = True

        globals()[
            "LLM_INFERENCE_VERIFIED"
        ] = True

        globals()[
            "MYRAA_LAST_INFERENCE"
        ] = result


        print()
        print(
            "Response:"
        )

        print("-" * 70)

        print(
            result["text"]
        )

        print("-" * 70)

        print(
            json.dumps(
                {
                    key: value
                    for key, value
                    in result.items()
                    if key != "text"
                },
                indent=2
            )
        )


        print()
        print(
            "MYRAA inference test: PASSED"
        )


    except Exception as exc:

        INFERENCE_AVAILABLE = False

        LLM_INFERENCE_VERIFIED = False

        MYRAA_LAST_INFERENCE = {

            "status":
                "failed",

            "error_type":
                type(exc).__name__,

            "error":
                str(exc)
        }


        globals()[
            "INFERENCE_AVAILABLE"
        ] = False

        globals()[
            "LLM_INFERENCE_VERIFIED"
        ] = False

        globals()[
            "MYRAA_LAST_INFERENCE"
        ] = MYRAA_LAST_INFERENCE


        print()
        print(
            "MYRAA inference FAILED."
        )

        print(
            f"{type(exc).__name__}: {exc}"
        )


print()
print("=" * 70)
print("CELL 8 STATUS")
print("=" * 70)

print(
    "Model loaded:",
    MODEL_LOADED
)

print(
    "Tokenizer loaded:",
    TOKENIZER_LOADED
)

print(
    "Inference available:",
    globals().get(
        "INFERENCE_AVAILABLE",
        False
    )
)

print(
    "LLM inference verified:",
    globals().get(
        "LLM_INFERENCE_VERIFIED",
        False
    )
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 10 — MYRAA JOB ENGINE
# JSON-SAFE / LOCAL JOB QUEUE
# ============================================================

import json
import uuid
from datetime import datetime, timezone
from pathlib import Path


# ============================================================
# SAFE CONFIG
# ============================================================

JOBS_DIR = Path(
    globals().get(
        "JOBS_DIR",
        "/kaggle/working/myraa_worker/jobs"
    )
)

JOBS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


WORKER_NAME = globals().get(
    "WORKER_NAME",
    "myraa-kaggle-t4x2"
)

PROVIDER = globals().get(
    "PROVIDER",
    "kaggle"
)


# ============================================================
# JSON SAFE
# ============================================================

def job_json_safe(value):

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):

        return {
            str(k): job_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):

        return [
            job_json_safe(v)
            for v in value
        ]

    if hasattr(value, "item"):

        try:
            return value.item()
        except Exception:
            pass

    if value is None:
        return None

    if isinstance(
        value,
        (str, int, float, bool)
    ):
        return value

    return str(value)


# ============================================================
# TIME
# ============================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


# ============================================================
# CREATE JOB
# ============================================================

def create_job(
    prompt,
    max_new_tokens=512
):

    if not isinstance(
        prompt,
        str
    ):

        raise TypeError(
            "prompt must be a string."
        )


    prompt = prompt.strip()


    if not prompt:

        raise ValueError(
            "prompt cannot be empty."
        )


    job_id = str(
        uuid.uuid4()
    )


    job = {

        "job_id":
            job_id,

        "worker":
            WORKER_NAME,

        "provider":
            PROVIDER,

        "type":
            "LLM_INFERENCE",

        "status":
            "QUEUED",

        "created_at":
            utc_now(),

        "input": {

            "prompt":
                prompt,

            "max_new_tokens":
                int(max_new_tokens)
        },

        "result":
            None,

        "error":
            None
    }


    job = job_json_safe(
        job
    )


    job_file = (
        JOBS_DIR
        / f"{job_id}.json"
    )


    with open(
        job_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            job,
            f,
            indent=2,
            ensure_ascii=False
        )


    return job_id


# ============================================================
# EXECUTE JOB
# ============================================================

def execute_job(
    job_id
):

    job_file = (
        JOBS_DIR
        / f"{job_id}.json"
    )


    if not job_file.exists():

        raise FileNotFoundError(
            f"Job not found: {job_id}"
        )


    with open(
        job_file,
        "r",
        encoding="utf-8"
    ) as f:

        job = json.load(f)


    job["status"] = "RUNNING"


    with open(
        job_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            job,
            f,
            indent=2,
            ensure_ascii=False
        )


    try:

        if "generate_response" not in globals():

            raise RuntimeError(
                "generate_response is unavailable. "
                "Run Cell 8 successfully first."
            )


        result = generate_response(

            job["input"]["prompt"],

            max_new_tokens=(
                job["input"]
                .get(
                    "max_new_tokens",
                    512
                )
            )
        )


        job["status"] = (
            "COMPLETED"
        )

        job["completed_at"] = (
            utc_now()
        )

        job["result"] = (
            job_json_safe(result)
        )

        job["error"] = None


    except Exception as exc:

        job["status"] = (
            "FAILED"
        )

        job["completed_at"] = (
            utc_now()
        )

        job["error"] = {

            "type":
                type(exc).__name__,

            "message":
                str(exc)
        }


    job = job_json_safe(
        job
    )


    with open(
        job_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            job,
            f,
            indent=2,
            ensure_ascii=False
        )


    return job


# ============================================================
# TEST JOB
# ============================================================

print("=" * 70)
print("MYRAA JOB ENGINE")
print("=" * 70)


if (
    "generate_response"
    in globals()
    and globals().get(
        "MODEL_LOADED",
        "model" in globals()
    )
):

    try:

        test_job_id = create_job(
            """
            You are MYRAA.
            Explain the difference between
            aggregate VRAM and contiguous VRAM
            in two short paragraphs.
            """
        )


        print(
            "Created job:",
            test_job_id
        )


        test_job = execute_job(
            test_job_id
        )


        print(
            json.dumps(
                test_job,
                indent=2,
                ensure_ascii=False
            )
        )


    except Exception as exc:

        print(
            "Job test failed:",
            type(exc).__name__,
            str(exc)
        )

else:

    print(
        "Job engine installed."
    )

    print(
        "Test execution skipped because "
        "LLM inference is not currently available."
    )


print("=" * 70)
print("CELL 10 COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 9 — GPU STATUS
# SELF-CONTAINED VERSION
# ============================================================

import json
import torch


def get_gpu_status():

    status = []

    # --------------------------------------------------------
    # CUDA check
    # --------------------------------------------------------

    if not torch.cuda.is_available():

        return {
            "cuda_available": False,
            "gpu_count": 0,
            "gpus": []
        }

    gpu_count = torch.cuda.device_count()

    # --------------------------------------------------------
    # Collect GPU information
    # --------------------------------------------------------

    for i in range(gpu_count):

        properties = (
            torch.cuda.get_device_properties(i)
        )

        total = (
            properties.total_memory
            / (1024 ** 3)
        )

        allocated = (
            torch.cuda.memory_allocated(i)
            / (1024 ** 3)
        )

        reserved = (
            torch.cuda.memory_reserved(i)
            / (1024 ** 3)
        )

        # ----------------------------------------------------
        # Approximate currently available VRAM
        #
        # This is based on reserved CUDA memory, so it is
        # an operational estimate rather than a hardware
        # guarantee.
        # ----------------------------------------------------

        free = max(
            0.0,
            total - reserved
        )

        status.append({

            "gpu": i,

            "name": (
                torch.cuda.get_device_name(i)
            ),

            "total_vram_gb": round(
                total,
                2
            ),

            "allocated_vram_gb": round(
                allocated,
                2
            ),

            "reserved_vram_gb": round(
                reserved,
                2
            ),

            "approx_free_vram_gb": round(
                free,
                2
            ),

            "compute_capability": (
                f"{properties.major}."
                f"{properties.minor}"
            )
        })

    return {
        "cuda_available": True,

        "gpu_count": gpu_count,

        "gpus": status
    }


# ============================================================
# GENERATE STATUS
# ============================================================

GPU_STATUS = get_gpu_status()


# ============================================================
# DISPLAY STATUS
# ============================================================

print("=" * 70)
print("MYRAA — GPU STATUS")
print("=" * 70)

print(
    json.dumps(
        GPU_STATUS,
        indent=2
    )
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 11 — FINAL MYRAA WORKER REPORT
# FULLY JSON-SAFE / STATE-AWARE / TRUTHFUL
# ============================================================

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import torch


print("=" * 70)
print("MYRAA KAGGLE WORKER FINAL REPORT")
print("=" * 70)


# ============================================================
# 1. UNIVERSAL JSON-SAFE CONVERTER
# ============================================================

def make_json_safe(value):

    # PyTorch dtype
    if isinstance(
        value,
        torch.dtype
    ):

        return str(
            value
        ).replace(
            "torch.",
            ""
        )


    # PyTorch device
    if isinstance(
        value,
        torch.device
    ):

        return str(
            value
        )


    # Path
    if isinstance(
        value,
        Path
    ):

        return str(
            value
        )


    # Dict
    if isinstance(
        value,
        dict
    ):

        return {
            str(key):
                make_json_safe(val)
            for key, val
            in value.items()
        }


    # List / tuple
    if isinstance(
        value,
        (list, tuple)
    ):

        return [
            make_json_safe(item)
            for item in value
        ]


    # Set
    if isinstance(
        value,
        set
    ):

        return [
            make_json_safe(item)
            for item in value
        ]


    # NumPy-style scalar
    if hasattr(
        value,
        "item"
    ):

        try:

            return value.item()

        except Exception:

            pass


    # Standard JSON types
    if value is None:

        return None


    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool
        )
    ):

        return value


    # Final fallback
    return str(
        value
    )


# ============================================================
# 2. IDENTITY
# ============================================================

WORKER_NAME = globals().get(
    "WORKER_NAME",
    "myraa-kaggle-t4x2"
)

WORKER_VERSION = globals().get(
    "WORKER_VERSION",
    "1.0.0"
)

PROVIDER = globals().get(
    "PROVIDER",
    "kaggle"
)


# ============================================================
# 3. OUTPUT DIRECTORY
# ============================================================

if "OUTPUT_DIR" in globals():

    OUTPUT_DIR = Path(
        OUTPUT_DIR
    )

else:

    OUTPUT_DIR = (
        Path.cwd()
        / "myraa_worker"
        / "reports"
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. HARDWARE
# ============================================================

CUDA_AVAILABLE = bool(
    torch.cuda.is_available()
)


GPU_COUNT = (
    torch.cuda.device_count()
    if CUDA_AVAILABLE
    else 0
)


GPU_INFO = []


AGGREGATE_VRAM = 0.0


if CUDA_AVAILABLE:

    for i in range(
        GPU_COUNT
    ):

        props = (
            torch.cuda
            .get_device_properties(i)
        )


        total_vram = (
            props.total_memory
            / (1024 ** 3)
        )


        AGGREGATE_VRAM += (
            total_vram
        )


        GPU_INFO.append({

            "index":
                int(i),

            "name":
                str(props.name),

            "vram_gb":
                round(
                    total_vram,
                    2
                ),

            "compute_capability":
                (
                    f"{props.major}."
                    f"{props.minor}"
                )
        })


# ============================================================
# 5. SOFTWARE
# ============================================================

NCCL_AVAILABLE = bool(
    torch.distributed
    .is_nccl_available()
)


PYTORCH_VERSION = str(
    torch.__version__
)


CUDA_VERSION = (
    str(torch.version.cuda)
    if torch.version.cuda is not None
    else None
)


# ============================================================
# 6. CAPABILITY FLAGS
#
# Read CURRENT GLOBALS first.
# Do NOT trust stale CAPABILITIES values.
# ============================================================

FP16_VERIFIED = bool(
    CUDA_AVAILABLE
    and GPU_COUNT >= 1
)


MULTI_GPU_CUDA_VERIFIED = bool(
    CUDA_AVAILABLE
    and GPU_COUNT >= 2
)


NCCL_COMMUNICATION_VERIFIED = bool(
    globals().get(
        "NCCL_COMMUNICATION_VERIFIED",
        False
    )
)


MODEL_SHARDING_VERIFIED = bool(
    globals().get(
        "MODEL_SHARDING_VERIFIED",
        False
    )
)


LLM_INFERENCE_VERIFIED = bool(
    globals().get(
        "LLM_INFERENCE_VERIFIED",
        False
    )
)


TRAINING_VERIFIED = bool(
    globals().get(
        "TRAINING_VERIFIED",
        False
    )
)


EMBEDDINGS_VERIFIED = bool(
    globals().get(
        "EMBEDDINGS_VERIFIED",
        False
    )
)


# ============================================================
# 7. MODEL INFORMATION
# ============================================================

MODEL_ID = str(
    globals().get(
        "MODEL_ID",
        "unknown"
    )
)


USE_4BIT = bool(
    globals().get(
        "USE_4BIT",
        False
    )
)


TORCH_DTYPE_RAW = globals().get(
    "TORCH_DTYPE",
    "float16"
)


if isinstance(
    TORCH_DTYPE_RAW,
    torch.dtype
):

    TORCH_DTYPE_NAME = (
        str(
            TORCH_DTYPE_RAW
        )
        .replace(
            "torch.",
            ""
        )
    )

else:

    TORCH_DTYPE_NAME = str(
        TORCH_DTYPE_RAW
    )


# ============================================================
# 8. BITSANDBYTES
# ============================================================

BITSANDBYTES_AVAILABLE = bool(
    globals().get(
        "BITSANDBYTES_AVAILABLE",
        False
    )
)


BITSANDBYTES_VERSION = (
    globals().get(
        "BITSANDBYTES_VERSION",
        None
    )
)


BITSANDBYTES_ERROR = (
    globals().get(
        "BITSANDBYTES_ERROR",
        None
    )
)


# ============================================================
# 9. GPU STATUS
# ============================================================

def get_gpu_status():

    status = []


    if not torch.cuda.is_available():

        return status


    for i in range(
        torch.cuda.device_count()
    ):

        properties = (
            torch.cuda
            .get_device_properties(i)
        )


        total = (
            properties.total_memory
            / (1024 ** 3)
        )


        allocated = (
            torch.cuda
            .memory_allocated(i)
            / (1024 ** 3)
        )


        reserved = (
            torch.cuda
            .memory_reserved(i)
            / (1024 ** 3)
        )


        free = max(
            0.0,
            total - reserved
        )


        status.append({

            "gpu":
                int(i),

            "name":
                str(
                    torch.cuda
                    .get_device_name(i)
                ),

            "total_vram_gb":
                round(
                    total,
                    2
                ),

            "allocated_vram_gb":
                round(
                    allocated,
                    2
                ),

            "reserved_vram_gb":
                round(
                    reserved,
                    2
                ),

            "approx_free_vram_gb":
                round(
                    free,
                    2
                ),

            "compute_capability":
                (
                    f"{properties.major}."
                    f"{properties.minor}"
                )
        })


    return status


# ============================================================
# 10. MODEL TEST RESULT
# ============================================================

MULTI_GPU_RESULT = (
    globals().get(
        "MYRAA_MULTI_GPU_RESULT",
        {}
    )
)


if not isinstance(
    MULTI_GPU_RESULT,
    dict
):

    MULTI_GPU_RESULT = {
        "raw":
            str(MULTI_GPU_RESULT)
    }


MULTI_GPU_RESULT = (
    make_json_safe(
        MULTI_GPU_RESULT
    )
)


# ============================================================
# 11. LAST INFERENCE
# ============================================================

LAST_INFERENCE = (
    globals().get(
        "MYRAA_LAST_INFERENCE",
        None
    )
)


LAST_INFERENCE = (
    make_json_safe(
        LAST_INFERENCE
    )
)


# ============================================================
# 12. BUILD FINAL FEATURES
# ============================================================

FINAL_FEATURES = {

    "fp16":
        FP16_VERIFIED,

    "nccl_installed":
        NCCL_AVAILABLE,

    "multi_gpu_cuda":
        MULTI_GPU_CUDA_VERIFIED,

    "nccl_communication":
        NCCL_COMMUNICATION_VERIFIED,

    "model_sharding":
        MODEL_SHARDING_VERIFIED,

    "llm_inference":
        LLM_INFERENCE_VERIFIED,

    "training":
        TRAINING_VERIFIED,

    "embeddings":
        EMBEDDINGS_VERIFIED
}


# ============================================================
# 13. FINAL REPORT
# ============================================================

FINAL_REPORT = {

    "schema_version":
        "1.1",

    "worker": {

        "name":
            WORKER_NAME,

        "version":
            WORKER_VERSION,

        "provider":
            PROVIDER
    },


    "hardware": {

        "cuda_available":
            CUDA_AVAILABLE,

        "gpu_count":
            GPU_COUNT,

        "gpus":
            GPU_INFO,

        "aggregate_vram_gb":
            round(
                AGGREGATE_VRAM,
                2
            ),

        "aggregate_vram_is_contiguous":
            False
    },


    "software": {

        "python":
            str(
                sys.version
            ),

        "pytorch":
            PYTORCH_VERSION,

        "cuda":
            CUDA_VERSION,

        "nccl_available":
            NCCL_AVAILABLE
    },


    "capabilities":
        FINAL_FEATURES,


    "model": {

        "id":
            MODEL_ID,

        "quantization":
            (
                "4bit"
                if USE_4BIT
                else "none"
            ),

        # IMPORTANT:
        # Always a STRING.
        "dtype":
            TORCH_DTYPE_NAME,

        "bitsandbytes_available":
            BITSANDBYTES_AVAILABLE,

        "bitsandbytes_version":
            BITSANDBYTES_VERSION,

        "bitsandbytes_error":
            BITSANDBYTES_ERROR
    },


    "verification": {

        "multi_gpu_test":
            MULTI_GPU_RESULT,

        "last_inference":
            LAST_INFERENCE
    },


    "gpu_status":
        get_gpu_status(),


    "generated_at":
        datetime.now(
            timezone.utc
        ).isoformat()
}


# ============================================================
# 14. FINAL RECURSIVE JSON-SAFETY PASS
# ============================================================

FINAL_REPORT = make_json_safe(
    FINAL_REPORT
)


# ============================================================
# 15. SERIALIZATION TEST
# ============================================================

try:

    FINAL_REPORT_JSON = json.dumps(
        FINAL_REPORT,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )

except Exception as exc:

    print()
    print("=" * 70)
    print("FINAL REPORT SERIALIZATION FAILED")
    print("=" * 70)

    print(
        "Error type:",
        type(exc).__name__
    )

    print(
        "Error:",
        exc
    )

    raise


# ============================================================
# 16. SAVE
# ============================================================

FINAL_REPORT_FILE = (
    OUTPUT_DIR
    / "myraa_worker_report.json"
)


with open(
    FINAL_REPORT_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        FINAL_REPORT_JSON
    )


# ============================================================
# 17. PUBLISH GLOBALS
# ============================================================

globals()[
    "FINAL_REPORT"
] = FINAL_REPORT


globals()[
    "FINAL_REPORT_FILE"
] = FINAL_REPORT_FILE


# ============================================================
# 18. DISPLAY
# ============================================================

print()

print(
    FINAL_REPORT_JSON
)

print()

print(
    "Report:"
)

print(
    FINAL_REPORT_FILE
)

print()

print("=" * 70)
print("MYRAA WORKER REPORT COMPLETE")
print("=" * 70)